In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [2]:
data = pd.read_excel('housing.xlsx')

In [4]:
data['income_cat'] = pd.cut(data['median_income'],
                                bins=[0,1.5,3.0,4.5,6,np.inf],
                                labels=[1,2,3,4,5])

In [5]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data['income_cat']):
    strat_train_set = data.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = data.loc[test_index].drop("income_cat", axis=1)

In [6]:
housing = strat_train_set.copy()

In [7]:
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

In [8]:
num_attributes = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attributes = ["ocean_proximity"]

In [9]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler()),
])

In [10]:
cat_pipeline = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])

In [11]:
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, cat_attributes),
])

In [12]:
housing_prepared = full_pipeline.fit_transform(housing)

In [13]:
print(housing_prepared)

[[-0.94135046  1.34743822  0.02756357 ...  0.          0.
   0.        ]
 [ 1.17178212 -1.19243966 -1.72201763 ...  0.          0.
   1.        ]
 [ 0.26758118 -0.1259716   1.22045984 ...  0.          0.
   0.        ]
 ...
 [-1.5707942   1.31001828  1.53856552 ...  0.          0.
   0.        ]
 [-1.56080303  1.2492109  -1.1653327  ...  0.          0.
   0.        ]
 [-1.28105026  2.02567448 -0.13148926 ...  0.          0.
   0.        ]]


In [17]:
#Linear regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_prediction = lin_reg.predict(housing_prepared)
lin_rmse = root_mean_squared_error(housing_labels,lin_prediction)
print("Linear Regression RMSE", lin_rmse)

Linear Regression RMSE 69050.56219504568


In [18]:
#Decision Tree regression
decision_tree_reg = DecisionTreeRegressor()
decision_tree_reg.fit(housing_prepared, housing_labels)
decision_tree_prediction = decision_tree_reg.predict(housing_prepared)
decision_tree_rmse = root_mean_squared_error(housing_labels, decision_tree_prediction)
print("Decision Tree Regression", decision_tree_rmse)

Decision Tree Regression 0.0


In [19]:
#Random forest regression
random_reg = RandomForestRegressor()
random_reg.fit(housing_prepared, housing_labels)
random_prediction = random_reg.predict(housing_prepared)
random_rmse = root_mean_squared_error(housing_labels, random_prediction)
print("Random forest regression", random_rmse)

Random forest regression 18495.354982900702
